# Model training


#### Initialize

In [2]:
%load_ext autoreload
%autoreload 2
from config.settings_data import DataSetsSettings

profile = "default"
settings = DataSetsSettings.build(profile)

### Experiment: Aquisition Parameter Detection

In [1]:
%load_ext tensorboard
%tensorboard --logdir lightning_logs

Reusing TensorBoard on port 6006 (pid 17648), started 1 day, 4:04:37 ago. (Use '!kill 17648' to kill it.)

In [3]:
from pathlib import Path

import lightning as L

from lightning.pytorch.callbacks import (
    EarlyStopping,
    ModelCheckpoint,
)

from lightning.pytorch.loggers import TensorBoardLogger

from utils.data_repository_utils import load_dataset_catalog

from utils.models.model_catalog import ModelCatalog
from utils.models.training.data_module import ImageClassificationDataModule
from utils.models.training.binary_classification_module import (
    BinaryClassificationModule,
)

from torchvision.transforms import (
    Compose,
    RandomRotation,
    RandomHorizontalFlip,
    ToTensor,
)

# ============================================================
# Augmentation
# ============================================================


train_transform = Compose([
#    RandomRotation(
#        degrees=90,
#    ),

 #   RandomHorizontalFlip(
 #       p=0.3,
 #   ),

    ToTensor(),
])

evaluation_transform = Compose([
    ToTensor(),
])


# ============================================================
# Experiment configuration
# ============================================================

EXPERIMENT_NAME = "pean_xlarge_top10"
MODEL_NAME = "pean_xlarge"
BATCH_SIZE = 4
NUM_WORKERS = 8

LEARNING_RATE = 1e-3
WEIGHT_DECAY = 0.0
GAMMA = 0.7

MAX_EPOCHS = 1000
PATIENCE = 25

SEED = 42


# ============================================================
# Reproducibility
# ============================================================

L.seed_everything(
    SEED,
    workers=True,
)


# ============================================================
# Dataset catalog
# ============================================================

dataset_catalog = load_dataset_catalog(
    settings.ap_detection_config
)

print(
    f"Loaded datasets: {len(dataset_catalog)}"
)

for dataset_item in dataset_catalog:
    print(
        dataset_item.dataset_name,
        dataset_item.task,
        dataset_item.version,
    )


# ============================================================
# Select dataset
# ============================================================

dataset = dataset_catalog.get(
    dataset_name="d110_s120_PA_acquisition_parameters_detection",
    task="ap-detection",
    version="v1.0.0",
)

training_csv = dataset.training.csv_path
validation_csv = dataset.validation.csv_path
test_csv = dataset.test.csv_path

data_root = dataset.data_root

#print("\nDataset:")
#print(dataset)

#print("\nTraining CSV:")
#print(training_csv)

#print("\nValidation CSV:")
#print(validation_csv)

#print("\nTest CSV:")
#print(test_csv)

#print("\nData root:")
#print(data_root)


# ============================================================
# Data module
# ============================================================

data_module = ImageClassificationDataModule(
    training_csv=training_csv,
    validation_csv=validation_csv,
    test_csv=test_csv,
    data_root=data_root,

    batch_size=BATCH_SIZE,
    test_batch_size=BATCH_SIZE,
    num_workers=NUM_WORKERS,

    channels=3,
    balance_training=True,

    train_transform=train_transform,
    evaluation_transform=evaluation_transform,

)


# ============================================================
# Model catalog
# ============================================================

model_catalog = ModelCatalog()

print("\nAvailable models:")

for model_name in model_catalog.available_models():
    print(f"  {model_name}")


# ============================================================
# PEIN Tiny / Top-10
# ============================================================

network = model_catalog.create(
    MODEL_NAME,

    in_channels=3,

    aggregation="gap",
    #top_k=10,
)

#print("\nModel:")
#print(network)


# ============================================================
# Parameter count
# ============================================================

num_parameters = sum(
    parameter.numel()
    for parameter in network.parameters()
    if parameter.requires_grad
)

print(
    f"\nTrainable parameters: "
    f"{num_parameters:,}"
)


# ============================================================
# Lightning binary classification module
# ============================================================

module = BinaryClassificationModule(
    model=network,

    learning_rate=LEARNING_RATE,
    optimizer_name="Adam",

    weight_decay=WEIGHT_DECAY,
    gamma=GAMMA,

    decision_threshold=0.5,
)


# ============================================================
# TensorBoard logger
# ============================================================

logger = TensorBoardLogger(
    save_dir="lightning_logs",
    name=EXPERIMENT_NAME,
)

print(
    "\nTensorBoard log directory:"
)

print(
    logger.log_dir
)


# ============================================================
# Checkpoint
# ============================================================

checkpoint_callback = ModelCheckpoint(
    dirpath=Path(logger.log_dir) / "checkpoints",

    filename=(
        "best-"
        "{epoch:03d}-"
        "{val_loss:.4f}-"
        "{val_auc:.4f}"
    ),

    monitor="val_loss",
    mode="min",

    save_top_k=1,
    save_last=True,
)


# ============================================================
# Early stopping
# ============================================================

early_stopping_callback = EarlyStopping(
    monitor="val_loss",
    mode="min",

    patience=PATIENCE,
)


# ============================================================
# Trainer
# ============================================================

trainer = L.Trainer(
    max_epochs=MAX_EPOCHS,

    accelerator="gpu",
    devices=1,

    logger=logger,

    callbacks=[
        checkpoint_callback,
        early_stopping_callback,
    ],

    log_every_n_steps=10,
)


# ============================================================
# Training
# ============================================================

trainer.fit(
    module,
    datamodule=data_module,
)


# ============================================================
# Best checkpoint information
# ============================================================

print(
    "\nBest checkpoint:"
)

print(
    checkpoint_callback.best_model_path
)

print(
    "Best validation loss:",
    checkpoint_callback.best_model_score,
)


# ============================================================
# Test using the best validation checkpoint
# ============================================================

test_results = trainer.test(
    module,
    datamodule=data_module,
    ckpt_path="best",
)

print(
    "\nTest results:"
)

print(
    test_results
)

W0822 18:50:59.414000 34924 Lib\site-packages\torch\utils\flop_counter.py:29] triton not found; flop counting will not work for triton kernels
Seed set to 42


Loaded datasets: 1
d110_s120_PA_acquisition_parameters_detection ap-detection v1.0.0

Available models:
  pean_large
  pean_medium
  pean_resnet101
  pean_resnet152
  pean_resnet18
  pean_resnet34
  pean_resnet50
  pean_resnext101_32x8d
  pean_resnext50_32x4d
  pean_small
  pean_tiny
  pean_wide_resnet101_2
  pean_wide_resnet50_2
  pean_xlarge

Trainable parameters: 1,000,000

TensorBoard log directory:
lightning_logs\pean_xlarge_top10\version_0


GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]


┏━━━┳━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━┳━━━━━━━┳━━━━━━━┓
┃   ┃ Name      ┃ Type              ┃ Params ┃ Mode  ┃ FLOPs ┃
┡━━━╇━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━╇━━━━━━━╇━━━━━━━┩
│ 0 │ model     │ PEAN              │  1.0 M │ train │     0 │
│ 1 │ criterion │ BCEWithLogitsLoss │      0 │ train │     0 │
└───┴───────────┴───────────────────┴────────┴───────┴───────┘

Trainable params: 1.0 M                                                                                            
Non-trainable params: 0                                                                                            
Total params: 1.0 M                                                                                                
Total estimated model params size (MB): 4.000                                                                      
Modules in train mode: 32                                                                                          
Modules in eval mode: 0                                                                                            
Total FLOPs: 0

Output()

... (2 outputs left out)

Restoring states from the checkpoint path at D:\000.Projects\Xray.Harmonization\lightning_logs\pean_xlarge_top10\version_0\checkpoints\best-epoch=016-val_loss=0.1581-val_auc=0.9779.ckpt
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0]
Loaded model weights from the checkpoint at D:\000.Projects\Xray.Harmonization\lightning_logs\pean_xlarge_top10\version_0\checkpoints\best-epoch=016-val_loss=0.1581-val_auc=0.9779.ckpt
c:\Users\dmitr\AppData\Local\pypoetry\Cache\virtualenvs\xray-harmonization-EktM-8ML-py3.11\Lib\site-packages\lightning\pytorch\utilities\_pytree.py:21: `isinstance(treespec, LeafSpec)` is deprecated, use `isinstance(treespec, TreeSpec) and treespec.is_leaf()` instead.


Output()

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━┓
┃        Test metric        ┃       DataLoader 0        ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━┩
│       test_accuracy       │    0.9237614870071411     │
│         test_auc          │    0.9646344184875488     │
│          test_fn          │           148.0           │
│          test_fp          │           109.0           │
│         test_loss         │    0.24609512090682983    │
│          test_tn          │          2385.0           │
│          test_tp          │           729.0           │
└───────────────────────────┴───────────────────────────┘


Test results:
[{'test_loss': 0.24609512090682983, 'test_accuracy': 0.9237614870071411, 'test_auc': 0.9646344184875488, 'test_tn': 2385.0, 'test_fp': 109.0, 'test_fn': 148.0, 'test_tp': 729.0}]
